# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. We'll also display the dataset name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

The Croissant schema organizes records by "record sets", and each record set contains fields (columns). We will list all record sets with their `@id`, and then for one, list its fields (columns) and their `@id`.


In [ ]:
# List all record sets and their @id and name
print("Record sets available in the dataset:")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"  @id: {rs.id: <58} name: {rs.name}")

# Pick the main record set (the one with protein table, usually the first or principal one)
main_record_set = record_sets[0]
print(f"\nMain record set selected: {main_record_set.id} ({main_record_set.name})")

print("\nFields (columns) in this record set:")
for field in main_record_set.fields:
    print(f"  @id: {field.id: <35} name: {field.name}  (dataType: {field.data_type})")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. The `@id`s for the record set and fields are used to reference each entity precisely. We'll demonstrate this with the main protein abundance table record set.


In [ ]:
# Extract data from each record set by @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rsid in record_set_ids:
    # Load records using the @id
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df

# Print available columns (by @id) for the main record set, and show the first 5 records
main_rs_id = record_sets[0].id
print(f"Fields in main record set (@id={main_rs_id}):\n", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering, normalization, and grouping. We will select a numeric field (for example, `mw` for molecular weight) and show how to filter and normalize it. We'll also group data by a categorical field like `description` if available.


In [ ]:
# Choose main record set and a numeric field by their @id
record_set_id = main_rs_id

# Display all columns to help user pick correct field for analysis
print("Available fields in main DataFrame:")
print(dataframes[record_set_id].columns.tolist())

# Let's pick a numeric field (e.g., 'mw' for molecular weight or 'peptide_count')
# Replace with actual column names if different
numeric_field = None
for col in dataframes[record_set_id].columns:
    # Look for common numeric field names
    if col.lower() in ['mw', 'molecular_weight', 'peptide_count', 'coverage', 'abundance']:
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = dataframes[record_set_id].columns[0]  # fallback

print(f"Selected numeric field for analysis: {numeric_field}")

# Remove outliers: filter numeric field > 10 (example threshold)
try:
    df = dataframes[record_set_id].copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (showing up to 5):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a string/categorical field (e.g., 'description', 'accession'), if present
    possible_group_fields = [c for c in df.columns if c.lower() in ['description', 'accession']]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}' (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No categorical field (e.g. 'description', 'accession') found for grouping.")
except Exception as e:
    print("Could not perform numeric analysis due to: ", e)

## 5. Visualization

Visualize the distribution of the selected numeric field and the normalized values. We'll use matplotlib to plot a histogram and a boxplot.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the filtered numeric field
if numeric_field in filtered_df.columns:
    plt.figure(figsize=(12, 4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")

    plt.subplot(1,2,2)
    sns.boxplot(x=filtered_df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field} (filtered)")
    plt.show()
    
    # If normalized column exists, plot it too
    norm_col = f"{numeric_field}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(filtered_df[norm_col].dropna(), kde=True)
        plt.title(f"Normalized {numeric_field} (Z-score)")
        plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion

In this notebook, we loaded the Mass Spectrometry Analysis of Extracellular Vesicles dataset using its Croissant schema, identified the record structure, and performed basic filtering and normalization on quantitative fields. These steps enabled an initial exploration and understanding of the dataset structure and its record contents, laying the groundwork for more specialized analyses such as protein abundance statistics, comparison, or clustering based on field values.
